# MECHA in metafspm, 12: from the legacy MECHA to the component

**For:** developers who know the original MECHA (`Mecha`, `InData`, its XML files) and switch to the metafspm
version. The original is kept, frozen, as `openalea.mecha.legacy` (PR #67's head, with the developers' last
additions): it is the reference the components are compared with, and it is not developed further. New physics goes
into the components (developer tutorial 5), with a comparison or a unit test and a row in `doc/legacy.rst` and
`doc/equations.rst`.

This tutorial runs **one root section with both**: the legacy's `Mecha` on a GRANAP organ, and the component in a
metafspm scene on the same organ; it maps each common XML input and option to its parameter, compares kr, the uptake
and every node's potential, and shows where the differences are registered. **Run time:** about a minute.

> **Developer tutorial 12** of thirteen:
> | | notebook | what it covers |
> |---|---|---|
> | 0 | `00_overview.ipynb` | the package's modules, the original MECHA and its reactions branch, how the port is validated, the docs |
> | 1 | `01_granap_mecha.ipynb` | GRANAP's anatomies and MECHA's hydraulics coupled in a plant model; MECHA's inputs by role; scenarios; the barriers |
> | 2 | `02_soil_gradient.ipynb` | the plant problem with a soil drier on one side of a root section; a root partly out of contact, by position or by cells |
> | 3 | `03_needle_solutes.ipynb` | a pine needle of three sections, its transpiration, MechaSolutes over a day |
> | 4 | `04_full_plant.ipynb` | a whole seedling in its soil, run to the fixed point |
> | 5 | `05_extend_the_component.ipynb` | add a variable and a process to the component, run and test it |
> | 6 | `06_solute_storage_and_decay.ipynb` | the solutes' storage fractions, their first-order loss and the liquid boundary |
> | 7 | `07_section_options.ipynb` | the section's hydraulic options: the barrier codes 0-11, φ-thickenings, sclerenchyma, the exodermis' own values, aquaporins, plasmodesmata rules, walls and air spaces, xylem pieces, tracheids, measured axial conductances |
> | 8 | `08_standard_problem.ipynb` | the standard problem's boundaries (soil sides, xylem and phloem pressures or flows, the elongation zone), the osmotic scenarios and the Casparian bands' σ, the flows by layer |
> | 9 | `09_solute_options.ipynb` | the solutes' pathways, schemes and time steps, held concentrations, their fluxes and steady states, the bands' σ, the coupled water and sugar |
> | 10 | `10_root_system_conductance.ipynb` | a root system's conductance (MARSHAL's Krs, SUF) upscaled from the anatomical network, and a two-parameter calibration on it |
> | 11 | `11_barriers_formalism.ipynb` | the barriers from the declarations: the walls' layers in parallel, the membranes' resistances in series, the bands' σ in the flow law and its Jacobian, the membranes' permeability from their structure |
> | 12 | `12_from_the_legacy.ipynb` | for developers of the original MECHA: one section run with the legacy `Mecha` and with the components, each XML input mapped to its parameter, the results compared |
>
> New to MECHA? Start with the user tutorials (`../user/01_first_run.ipynb`): they solve a root section and explain
> its results without the framework's details.

## 1. Two ways to the same section

| | the legacy | the component |
|---|---|---|
| the anatomy | a cell set (XML) or a GRANAP organ, read by `NetworkBuilder` | GRANAP's component (`GranapAnatomy`) writing the section into the plant graph (MPG) |
| the parameters | `InData`: its XML files (`Hydraulics.xml`, `Geometry.xml`, `BCs.xml`, `Hormones.xml`), or its attributes set in code | each component's declared parameters, per plant, in the plant's **scenario** |
| a stage of maturity | `geometry.set_maturity_stages([barrier, ...], height=[...])`, one solve each | the segment's `barrier` input (a development model: here the stand-in `Development`) and its `length` |
| the solve | `Mecha.solve_W(h, i_maturity)` (or `water_flux()`: every stage and scenario) | the rate and graph-system steps of `MechaHydraulics`, called by the Scene each step |
| the results | `Mecha`'s arrays (`kr_tot`, `total_flow`, the solution vector by legacy index) | variables on the DataStructure (`kr`, `standard_uptake`, `standard_psi` by node) |

The component's scene, one root segment of 200 µm (the legacy's default height), at barrier 1:

In [1]:
import sys

import numpy as np
import pandas as pd

from openalea.metafspm import CompositeModel, Planting, planting, Scene

from openalea.granap.metafspm.architectures import RootAxis
from openalea.granap.metafspm.component import GranapAnatomy, junction_rules
from openalea.granap.metafspm.library import template_by_index
from openalea.mecha.component import MechaHydraulics
from openalea.mecha.geometry import section_parts
from openalea.mecha.legacy import InData, Mecha, NetworkBuilder
from openalea.mecha.stand_ins import Development

sys.path.insert(0, "../../../test/comparison")
from legacy_run import LEGACY_OPTIONS                 # the port's options reproducing the legacy's choices


class RootSection(CompositeModel):                     # the first tutorial's plant model
    initiators = (RootAxis, GranapAnatomy)
    from_scale, nodes, wiring = "SubOrgan", "Compartment", junction_rules

    parts = {"anatomy": GranapAnatomy, "development": Development, "hydraulics": MechaHydraulics}
    schedule = ("development", "anatomy", "hydraulics")


def component_run(barrier, **parameters):
    """The component on one root segment at *barrier*, MECHA's *parameters* in the scenario: (its DataStructure,
    its MechaHydraulics), solved once."""
    scene = Scene(planting([Planting()], RootSection, dict(parameters, mature_barrier=float(barrier))))
    scene.run()
    population = scene.populations[0]
    return population.data_structure, population.instance.hydraulics


data_structure, mecha = component_run(1)
segment = int(data_structure.entity_ids("SubOrgan")[0])
height = float(np.asarray(data_structure.get("length"))[0]) * 1.0E6              # µm
print(f"segment {segment}, {height:.0f} µm long; kr {float(data_structure.get('kr')[0]):.6g} cm hPa-1 d-1")

segment 16, 200 µm long; kr 0.000107298 cm hPa-1 d-1


The legacy reads the **same GRANAP organ**: GRANAP's component keeps each generated anatomy as a template of the
population (`template_by_index`), its organ the one GRANAP's legacy exports write. `InData()` without files gives
the legacy's defaults; its attributes are set in code as its own examples do (`example/legacy/root_example.py`). One
change to them: the phloem-pole plasmodesmata factor (PPP), 0 by default, isolates the companion cells once
`develop`'s factors are applied (`doc/legacy.rst`, d2): set to 1, as in every comparison.

In [2]:
organ = template_by_index(data_structure.mtg, int(np.asarray(data_structure.get("anatomy_template"))[0])).organ
organ.export_to_adjencymatrix()                      # GRANAP's legacy export: the networkx graph the legacy reads


def legacy_run(barriers, **hydraulic):
    """The legacy Mecha on the organ, one maturity stage per code of *barriers* (200 µm high), its HydraulicData's
    attributes *hydraulic* set: (the Mecha, its NetworkBuilder, the solution of each stage)."""
    data = InData()
    for entry in data.hydraulic.kpl:
        entry["phloem_pericycle_pole_factor"] = 1.0
    for name, value in hydraulic.items():
        setattr(data.hydraulic, name, value)
    data.geometry.set_maturity_stages(list(barriers), height=[height] * len(barriers))
    network = NetworkBuilder(organ)
    network.populate_from_network()
    legacy = Mecha(data, network=network)
    solutions = [np.asarray(legacy.solve_W(h=0, i_maturity=i)[0]).reshape(-1) for i in range(len(barriers))]
    return legacy, network, solutions


legacy, network, solutions = legacy_run([1, 3])

Setting default boundary conditions...


Solving system
0.029377823999766406 seconds process time to solve system


Solving system
0.03079389499998797 seconds process time to solve system


## 2. kr and the uptake

The component with its own defaults, then with `LEGACY_OPTIONS` (`test/comparison/legacy_run.py`), the options that
reproduce the legacy where it departs from the original MECHA or where the port fixed a problem of it:

In [3]:
pd.Series(LEGACY_OPTIONS, name="value").to_frame()

,value
plasmodesmata_rules,2.0
kpl_PCC_factor,0.0
kpl_PST_factor,0.0
standard_axial_source,2.0
thickness_source,1.0
membrane_floor,2.0
k_xyl,1.0
k_sieve,1.0
sclerenchyma_plasmodesmata_rule,2.0
lamella_wall_half,2.0


| option | the legacy's choice it reproduces |
|---|---|
| `plasmodesmata_rules` 2, `kpl_PCC_factor` 0, `kpl_PST_factor` 0 | the refactor's plasmodesmata factor map (the original MECHA's rules are 1, the default) |
| `standard_axial_source` 2, `k_xyl`, `k_sieve` 1 | its xylem boundary prescribed through penalty conductances (ME-32) |
| `thickness_source` 1 | one wall thickness for every wall (the port's default: each GRANAP wall its own) |
| `membrane_floor` 2 | its membranes without aquaporins kept at `kmb` (ME-33) |
| `sclerenchyma_plasmodesmata_rule` 2 | the product of two fibres' factors on their plasmodesma (d17, U1) |
| `lamella_wall_half` 2, `membrane_surface_rule` 2 | `kw` in every membrane's wall half with the lamella model (d19, U9); its permeability's surface (d20, U6) |
| `standard_psi_sieve` NaN | no phloem boundary at barrier 0 without `BCs.xml` |

In [4]:
rows = {}
for barrier, i in ((1, 0), (3, 1)):
    own, _ = component_run(barrier)
    reproduced, _ = component_run(barrier, **LEGACY_OPTIONS)
    rows[barrier] = {"component, its defaults": float(own.get("kr")[0]),
                     "component, LEGACY_OPTIONS": float(reproduced.get("kr")[0]),
                     "legacy": float(legacy.kr_tot[i, 0])}
kr = pd.DataFrame(rows).T.rename_axis("barrier")
kr["relative difference (LEGACY_OPTIONS)"] = kr["component, LEGACY_OPTIONS"] / kr["legacy"] - 1
kr

,"component, its defaults","component, LEGACY_OPTIONS",legacy,relative difference (LEGACY_OPTIONS)
barrier,,,,
1,0.000107,0.000105,0.000105,-7.105427e-15
3,0.000041,0.000041,0.000041,4.107825e-14


With `LEGACY_OPTIONS`, kr is the legacy's to round-off; with its own defaults the component differs by up to a few per cent (3 % at barrier 1)
(mostly each wall's own thickness and the original plasmodesmata rules). The uptake, `standard_uptake` against the
legacy's `total_flow`, and every node's water potential: the legacy numbers its nodes walls, junctions, cells; the
port keeps GRANAP's vertex of each wall and cell (`mecha_index`, the legacy's index, is a property of the MPG) and
its junctions are matched by position:

In [5]:
data_structure, mecha = component_run(1, **LEGACY_OPTIONS)
nodes, _ = section_parts(data_structure)[segment]
mecha_index = data_structure.mtg.property("mecha_index")
legacy_index = np.array([int(mecha_index[int(v)]) for v in data_structure.entity_ids("node")[nodes]])
geometry = mecha.section_geometries()[segment][2]
by_position = {tuple(network.graph.nodes[j]["position"]): j for j in range(network.n_walls, network.n_wall_junction)}
for v in geometry.junctions:
    legacy_index[v] = by_position[(geometry.x[v], geometry.y[v])]
psi_component = np.asarray(data_structure.get("standard_psi"))[nodes]
psi_legacy = solutions[0][legacy_index]
print(f"uptake: component {float(data_structure.get('standard_uptake')[0]):.10g}, "
      f"legacy {float(np.asarray(legacy.total_flow[0][0]).reshape(-1)[0]):.10g} cm3 d-1")
print(f"{nodes.size} nodes; largest difference of the water potentials {np.abs(psi_component - psi_legacy).max():.1e} hPa "
      f"(range {np.ptp(psi_legacy):.0f} hPa)")

uptake: component 0.002631157151, legacy 0.002631157151 cm3 d-1
8269 nodes; largest difference of the water potentials 1.4e-10 hPa (range 5000 hPa)


## 3. A PR #67 feature on both sides

The suberin lamella (PR #67's `suberin_model='lamella'`), at barrier 3, each barrier's own conductivity. The legacy
reads them from its `kw_barrier` entries (`Hydraulics.xml`'s `kw_barrier` element: `Casp`, `Sub`, `Sub_in`,
`Sub_out`...) and its layer thicknesses; the component from its parameters:

In [6]:
legacy_barrier = {"value": 1.0E-16, "Casp": 1.0E-9, "Sub": 7.0E-9, "Sub_in": 3.0E-9, "Sub_out": 5.0E-9, "Lig": 1.0E-16}
legacy_lamella, _, _ = legacy_run([3], kw_barrier_elems=[legacy_barrier], suberin_model="lamella",
                                  sec_wall_thickness=0.5, tert_wall_thickness=1.0)
component_lamella, _ = component_run(3, **LEGACY_OPTIONS, kw_casparian=1.0E-9, kw_suberin_radial=7.0E-9,
                                     kw_suberin_in=3.0E-9, kw_suberin_out=5.0E-9, suberin_model=1.,
                                     secondary_wall_thickness=0.5, tertiary_wall_thickness=1.0)
print(f"kr: legacy {legacy_lamella.kr_tot[0, 0]:.10g}, component {float(component_lamella.get('kr')[0]):.10g}")

Setting default boundary conditions...


Solving system
0.04221221499983585 seconds process time to solve system


kr: legacy 5.241965757e-05, component 5.241965757e-05


## 4. The inputs and their parameters

The common inputs of the legacy's XML files and of its `HydraulicData` attributes (PR #67's included), with the
component's parameter (MechaHydraulics' unless said; MechaOsmoticPresets' and MechaSolutes' named). In the
component, each is set in the plant's scenario, `{"kw": 2.4e-4, ...}`, per plant:

| legacy (XML element or attribute) | component | |
|---|---|---|
| `Geometry.xml`: maturity `Barrier`, `height` | the segment's `barrier` (a development model's: `Development`'s `mature_barrier` here) and `length` | inputs |
| `Geometry.xml`: `thickness`; `use_wall_thickness` | `thickness`, `thickness_source` (2 each wall's own, the default) | |
| `Geometry.xml`: `Xylem_pieces`, `kInterC`, `PD_section` | `xylem_pieces`, `k_interc`, `pd_section` | |
| `Geometry.xml`: `passage_cell_range`, `aerenchyma_range` (cell ids) | GRANAP's tissues (`passage`, air spaces) | the anatomy's |
| `Hydraulics.xml`: `kw`, `kw_septa`, `km` | `kw`, `kw_septa`, `kmb` | |
| `kw_barrier`: `Casp`, `Sub_in`, `Sub_out`, `Sub`, `Lig` | `kw_casparian`, `kw_suberin_in`, `kw_suberin_out`, `kw_suberin_radial`, `kw_lignin` | |
| `kw_barrier`: `Casp_exo`, `Sub_exo`, `Lig_scl`, `phi_thick` | `kw_casparian_exodermis`, `kw_suberin_exodermis`, `kw_sclerenchyma` (or `kw_sclerenchyma_fraction`), `kw_phi_thickening` (or `kw_phi_thickening_fraction`) | |
| `kAQP` and its `*_factor`s, `ratio_cortex` | `kaqp`, `kaqp_endo_factor`, `kaqp_exo_factor`, `kaqp_cortex_factor`, `kaqp_stele_factor`..., `ratio_cortex` | |
| `aqp_factor`, `aqp_factor_by_type` (per cell id or type) | the node input `aquaporin_factor` (role "Regulation") | an input |
| `kw_wall_factor`, `kw_wall_factor_fn` | the node input `wall_conductivity_factor` (role "Anatomy") | an input |
| `Kpl` and its factors, `Fplxheight` | `kpl`, `kpl_*_factor` (`kpl_exodermis_factor`, `kpl_epidermis_factor`...), `fplxheight_*`, `plasmodesmata_rules` | |
| dead sclerenchyma (PR #67's draw, 0.8, × 0.05) | `sclerenchyma_dead_fraction`, `sclerenchyma_plasmodesmata_factor`, `sclerenchyma_seed`; the node input `living_cell` | |
| `suberin_model`, `prim_wall_thickness`, `sec_wall_thickness`, `tert_wall_thickness` | `suberin_model` (0, 1), `primary_wall_thickness`, `secondary_wall_thickness`, `tertiary_wall_thickness` | |
| `sigma_casp_endo`, `sigma_casp_exo` | `sigma_casparian_endodermis`, `sigma_casparian_exodermis` | MechaOsmoticPresets |
| `Xcontactrange`; `Contactrange` (cells) | `x_contact`; the node input `soil_contact_cell` | |
| `K_xyl_spec`, `Kax_source` | Poiseuille's by default; the node input `measured_axial_conductivity` | |
| `BCs.xml`: `psi_soil_left`, `psi_soil_right`, `pressure_xyl_prox` (or a flow), the phloem's | `standard_psi_soil`, `standard_psi_soil_right`, `standard_psi_xylem` (`standard_axial_condition`, `standard_axial_flow`), `standard_psi_sieve`... | the standard problem |
| `BCs.xml`: the osmotic scenarios (`Os_hetero`, `s_hetero`, `s_factor`, `Os_cortex`, the soil's and the xylem's solutions) | `os_hetero`, `s_hetero`, `s_factor`, `os_cortex`, `osmotic_*`; `standard_osmotic` 1 | MechaOsmoticPresets |
| the elongation (`Elong_cell`, side difference, water fractions) | `standard_elongation`, `elongation_midpoint_rate`, `elongation_side_rate_difference`, `water_fraction_*` | |
| `SoluteTransport`: `apo_wall`, `plasmodesmata`, `membrane`, `membrane_pd`, `k_deg`, `conserve_boundary`, `c_boundary`, the Dirichlet nodes | `d_apoplast`, `d_plasmodesmata`, `d_membrane`, `membrane_lipid_permeability`, `membrane_aquaporin_permeability`, `k_decay`, `solute_boundary_advection`, `c_boundary`, the node input `fixed_sucrose` | MechaSolutes |
| `SoluteTransport.boundary_flux` | the node state `held_solute_flux` | MechaSolutes |

`openalea.mecha.simulation.parameters()` lists every parameter with its unit, description and bounds; developer
tutorial 1 lists the inputs by role.

## 5. Where the differences are written down

`doc/legacy.rst` (the docs' *legacy* page) is the map of the two versions:

- **The legacy's chain and the port's modules**: which legacy class became which component, step or function.
- **Coverage**: every input, option, boundary condition, solver path and output of the legacy, with the
  component's equivalent and the test comparing them (`test/comparison/`).
- **The registry**: each behaviour of the legacy met during the port, with what the port does: *ME-n* (the original
  MECHA's), *d1–d22* (the `develop` branch's additions, PR #67's among them: d12 per-wall thickness, d13–d16 the
  barriers' values and the walls lining air spaces, d17 the dead sclerenchyma, d18 the aquaporin and wall factors,
  d19 the lamella, d20 the membranes' permeability, d21 the bands' σ, d22 the flux through the held nodes), *r1–r12*
  (the reactions branch's), *U1–U9* (PR #67's problems, registered where the port departs).

A difference between the two versions is either a row there (and, when the legacy's choice is kept for the
comparisons, an entry of `LEGACY_OPTIONS`) or a bug to report. The comparisons run with
`pytest test/comparison -q` (about ten minutes); `example/comparison/` draws them as figures.

## Try it yourself

1. Set the legacy's `kw` (`data.hydraulic.kw`) and the component's `kw` to 1.2 × 10⁻⁴ (and `kw_septa` below): do
   they still agree?
2. Remove one entry of `LEGACY_OPTIONS` at a time: which one changes kr most at barrier 1? Find its row in
   `doc/legacy.rst`.
3. Run the legacy's `water_flux()` with an osmotic scenario (`data.boundary.add_scenario`, as `root_example.py`) and
   the component with MechaOsmoticPresets (developer tutorial 8): compare the uptakes.